# Phase 2 Manual Validation: Framework Mode Separation

This notebook validates end-to-end separation of `exploratory`, `locked_comparison`, and `confirmatory` paths by executing real CLIs and inspecting artifacts.

## A) Setup and Purpose
Manual validation of exploratory, locked comparison, and confirmatory workflow separation via real CLIs.


## Sections
A. Setup and path resolution  
B. Preflight + CLI fallback resolution  
C. Helpers  
D. Exploratory validation  
E. Locked comparison validation (dry + real)  
F. Confirmatory validation (dry + real)  
G. Cross-mode summary + guardrail checks + final verdict

## B) Setup / Path Resolution and Preflight
Resolve repo-relative paths, output roots, environment defaults, and CLI fallback commands.


In [3]:
from __future__ import annotations
import csv, json, os, shlex, subprocess, sys
from collections import Counter
from datetime import datetime
from pathlib import Path
try:
    import pandas as pd
except Exception:
    pd = None

def repo_root(start: Path|None=None)->Path:
    p=(start or Path.cwd()).resolve()
    for c in [p,*p.parents]:
        if (c/'pyproject.toml').exists() and (c/'src'/'Thesis_ML').exists(): return c
    raise FileNotFoundError('repo root not found')

ROOT=repo_root()
PROTOCOL=ROOT/'configs'/'protocols'/'thesis_canonical_v1.json'
COMPARISON=ROOT/'configs'/'comparisons'/'model_family_comparison_v1.json'
SID=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=ROOT/'outputs'/'manual_validation'/'framework_modes'/f'session_{SID}'
EXP=OUT/'exploratory'; CMP_DRY=OUT/'comparisons'/'dry_run'; CMP_REAL=OUT/'comparisons'/'real'; PROT_DRY=OUT/'confirmatory'/'dry_run'; PROT_REAL=OUT/'confirmatory'/'real'
for p in [EXP,CMP_DRY,CMP_REAL,PROT_DRY,PROT_REAL]: p.mkdir(parents=True,exist_ok=True)

assert PROTOCOL.exists(), f'missing protocol: {PROTOCOL}'
assert COMPARISON.exists(), f'missing comparison: {COMPARISON}'
INDEX=Path(os.environ.get('THESIS_ML_INDEX_CSV', ROOT/'Data'/'processed'/'dataset_index.csv')).resolve()
DATA=Path(os.environ.get('THESIS_ML_DATA_ROOT', ROOT/'Data')).resolve()
CACHE=Path(os.environ.get('THESIS_ML_CACHE_DIR', ROOT/'Data'/'processed'/'feature_cache')).resolve()
assert INDEX.exists(), f'missing index: {INDEX}'
assert DATA.exists(), f'missing data root: {DATA}'
assert CACHE.exists(), f'missing cache: {CACHE}'

def env():
    e=os.environ.copy(); src=str(ROOT/'src'); e['PYTHONPATH']=src if not e.get('PYTHONPATH') else src+os.pathsep+e['PYTHONPATH']; return e

def probe(cmd):
    try: return subprocess.run(cmd,cwd=str(ROOT),env=env(),capture_output=True,text=True),None
    except FileNotFoundError as ex: return None,str(ex)

def resolve(cli,module):
    p,e=probe([cli,'--help'])
    if p is not None and p.returncode==0: print('Using CLI:',cli); return [cli]
    m=[sys.executable,'-m',module]; q,f=probe(m+['--help'])
    if q is not None and q.returncode==0: print('Using module:', ' '.join(m)); return m
    raise RuntimeError(f'Cannot resolve {cli}/{module} CLI: {(p.stderr if p else e)} MOD: {(q.stderr if q else f)}')

EXP_CMD=resolve('thesisml-run-experiment','Thesis_ML.experiments.run_experiment')
CMP_CMD=resolve('thesisml-run-comparison','Thesis_ML.cli.comparison_runner')
PROT_CMD=resolve('thesisml-run-protocol','Thesis_ML.cli.protocol_runner')

def run(cmd,check=False):
    print('$',' '.join(shlex.quote(x) for x in cmd)); r=subprocess.run(cmd,cwd=str(ROOT),env=env(),capture_output=True,text=True)
    print('rc=',r.returncode); print('---stdout---'+(r.stdout.strip() or '<empty>')); print('---stderr---'+(r.stderr.strip() or '<empty>'))
    if check and r.returncode!=0: raise AssertionError(f'command failed rc={r.returncode}')
    return {'rc':r.returncode,'out':r.stdout.strip(),'err':r.stderr.strip()}

def j(path):
    d=json.loads(Path(path).read_text(encoding='utf-8')); assert isinstance(d,dict); return d

def rows(path):
    with Path(path).open('r',encoding='utf-8',newline='') as h: return list(csv.DictReader(h))

def need(base,names):
    miss=[n for n in names if not (Path(base)/n).exists()]; assert not miss,f'missing in {base}: {miss}'

def tree(root,max_depth=3,max_entries=120):
    root=Path(root); rd=len(root.parts); c=0; print('Tree:',root)
    for p in sorted(root.rglob('*')):
        d=len(p.parts)-rd
        if d>max_depth: continue
        print('  '*d+'- '+str(p.relative_to(root))+('/' if p.is_dir() else '')); c+=1
        if c>=max_entries: print('...truncated'); break

def parse_out(out):
    x=json.loads(out); assert isinstance(x,dict); return x

def subjects(idx):
    s=set();
    with Path(idx).open('r',encoding='utf-8',newline='') as h:
        for r in csv.DictReader(h):
            v=str(r.get('subject','')).strip();
            if v: s.add(v)
    y=sorted(s); assert y,'no subjects'; return y

MODE={}; CHECK={}
print('ROOT=',ROOT); print('OUT=',OUT)

Using CLI: thesisml-run-experiment
Using module: d:\Khaled\Projects\VScode\Thesis\.venv\Scripts\python.exe -m Thesis_ML.cli.comparison_runner
Using module: d:\Khaled\Projects\VScode\Thesis\.venv\Scripts\python.exe -m Thesis_ML.cli.protocol_runner
ROOT= D:\Khaled\Projects\VScode\Thesis
OUT= D:\Khaled\Projects\VScode\Thesis\outputs\manual_validation\framework_modes\session_20260315_031435


## C) Exploratory Mode Validation
Run one small exploratory command and verify run artifact mode metadata.


In [4]:
# Exploratory mode validation
subj=subjects(INDEX)[0]
run_id=f'manual_exploratory_{subj}_{SID}'
cmd=[*EXP_CMD,'--index-csv',str(INDEX),'--data-root',str(DATA),'--cache-dir',str(CACHE),'--target','coarse_affect','--model','ridge','--cv','within_subject_loso_session','--subject',subj,'--seed','42','--run-id',run_id,'--reports-root',str(EXP)]
rex=run(cmd,check=True); _=parse_out(rex['out'])
run_dir=EXP/run_id
need(run_dir,['config.json','metrics.json','fold_metrics.csv','fold_splits.csv','predictions.csv','spatial_compatibility_report.json','interpretability_summary.json','run_status.json'])
cfg=j(run_dir/'config.json'); met=j(run_dir/'metrics.json')
assert cfg.get('framework_mode')=='exploratory' and met.get('framework_mode')=='exploratory'
assert cfg.get('canonical_run') is False and met.get('canonical_run') is False
assert cfg.get('protocol_id') in (None,'') and cfg.get('comparison_id') in (None,'')
MODE['exploratory']={'mode':'exploratory','command':' '.join(shlex.quote(x) for x in cmd),'rc':rex['rc'],'output_root':str(EXP),'report_dir':str(run_dir),'framework_mode':cfg.get('framework_mode'),'canonical_run':cfg.get('canonical_run'),'protocol_meta':bool(cfg.get('protocol_id')),'comparison_meta':bool(cfg.get('comparison_id')),'manifests':'n/a'}
CHECK['exploratory']=True
tree(run_dir,2)

$ thesisml-run-experiment --index-csv 'D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv' --data-root 'D:\Khaled\Projects\VScode\Thesis\Data' --cache-dir 'D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache' --target coarse_affect --model ridge --cv within_subject_loso_session --subject sub-001 --seed 42 --run-id manual_exploratory_sub-001_20260315_031435 --reports-root 'D:\Khaled\Projects\VScode\Thesis\outputs\manual_validation\framework_modes\session_20260315_031435\exploratory'
rc= 0
---stdout---{
  "results": [
    {
      "model": "ridge",
      "run_id": "manual_exploratory_sub-001_20260315_031435",
      "report_dir": "D:\\Khaled\\Projects\\VScode\\Thesis\\outputs\\manual_validation\\framework_modes\\session_20260315_031435\\exploratory\\manual_exploratory_sub-001_20260315_031435",
      "accuracy": 0.7128005198180637,
      "balanced_accuracy": 0.6705653021442496,
      "macro_f1": 0.6669815174750768
    }
  ]
}
---stderr---INFO Thesis_ML.features.nift

## D) Locked Comparison Mode Validation
Run comparison dry-run, then one real small variant chosen from compiled counts.


In [ ]:
# Locked comparison mode validation (dry-run then real smallest variant)
cmp_files=['comparison.json','compiled_comparison_manifest.json','comparison_summary.json','execution_status.json','report_index.csv']
dry=[*CMP_CMD,'--comparison',str(COMPARISON),'--all-variants','--reports-root',str(CMP_DRY),'--dry-run']
rd=run(dry,check=True); pd=parse_out(rd['out']); cmp_d=Path(pd['comparison_output_dir']); need(cmp_d,cmp_files)
vc=Counter(r['variant_id'] for r in rows(cmp_d/'report_index.csv')); variant=sorted(vc.items(),key=lambda kv:(kv[1],kv[0]))[0][0]
print('variant counts=',dict(vc),'selected=',variant)
real=[*CMP_CMD,'--comparison',str(COMPARISON),'--variant',variant,'--reports-root',str(CMP_REAL)]
rr=run(real,check=True); pr=parse_out(rr['out']); cmp_r=Path(pr['comparison_output_dir']); need(cmp_r,cmp_files)
rws=[r for r in rows(cmp_r/'report_index.csv') if r.get('status')=='completed']; assert rws,'no completed comparison rows'
r0=rws[0]; rdir=Path(r0['report_dir']) if r0.get('report_dir') else (CMP_REAL/Path(r0['run_id']))
need(rdir,['config.json','metrics.json','fold_metrics.csv','fold_splits.csv','predictions.csv','spatial_compatibility_report.json','interpretability_summary.json'])
cfg=j(rdir/'config.json'); met=j(rdir/'metrics.json')
assert cfg.get('framework_mode')=='locked_comparison' and met.get('framework_mode')=='locked_comparison'
assert cfg.get('canonical_run') is False and met.get('canonical_run') is False
assert cfg.get('comparison_id') and cfg.get('comparison_version') and cfg.get('comparison_variant_id')
assert cfg.get('protocol_id') in (None,'')
MODE['locked_comparison']={'mode':'locked_comparison','command':' '.join(shlex.quote(x) for x in real),'rc':rr['rc'],'output_root':str(CMP_REAL),'report_dir':str(rdir),'framework_mode':cfg.get('framework_mode'),'canonical_run':cfg.get('canonical_run'),'protocol_meta':bool(cfg.get('protocol_id')),'comparison_meta':bool(cfg.get('comparison_id')),'manifests':', '.join([f for f in cmp_files if (cmp_r/f).exists()])}
CHECK['comparison']=True
tree(cmp_r,3)

$ 'd:\Khaled\Projects\VScode\Thesis\.venv\Scripts\python.exe' -m Thesis_ML.cli.comparison_runner --comparison 'D:\Khaled\Projects\VScode\Thesis\configs\comparisons\model_family_comparison_v1.json' --all-variants --reports-root 'D:\Khaled\Projects\VScode\Thesis\outputs\manual_validation\framework_modes\session_20260315_031435\comparisons\dry_run' --dry-run
rc= 0
---stdout---{
  "comparison_id": "model-family-within-subject",
  "comparison_version": "1.0.0",
  "comparison_output_dir": "D:\\Khaled\\Projects\\VScode\\Thesis\\outputs\\manual_validation\\framework_modes\\session_20260315_031435\\comparisons\\dry_run\\comparison_runs\\model-family-within-subject__1.0.0",
  "n_completed": 0,
  "n_failed": 0,
  "n_planned": 6,
  "artifact_paths": {
    "comparison_json": "D:\\Khaled\\Projects\\VScode\\Thesis\\outputs\\manual_validation\\framework_modes\\session_20260315_031435\\comparisons\\dry_run\\comparison_runs\\model-family-within-subject__1.0.0\\comparison.json",
    "compiled_comparison_

## E) Confirmatory Mode Validation
Run protocol dry-run, then one real small suite chosen from compiled counts.


In [ ]:
# Confirmatory mode validation (dry-run then real smallest suite)
prot_files=['protocol.json','compiled_protocol_manifest.json','claim_to_run_map.json','suite_summary.json','execution_status.json','report_index.csv']
dry=[*PROT_CMD,'--protocol',str(PROTOCOL),'--all-suites','--reports-root',str(PROT_DRY),'--dry-run']
rd=run(dry,check=True); pd=parse_out(rd['out']); prot_d=Path(pd['protocol_output_dir']); need(prot_d,prot_files)
sc=Counter(r['suite_id'] for r in rows(prot_d/'report_index.csv')); suite=sorted(sc.items(),key=lambda kv:(kv[1],kv[0]))[0][0]
print('suite counts=',dict(sc),'selected=',suite)
real=[*PROT_CMD,'--protocol',str(PROTOCOL),'--suite',suite,'--reports-root',str(PROT_REAL)]
rr=run(real,check=True); pr=parse_out(rr['out']); prot_r=Path(pr['protocol_output_dir']); need(prot_r,prot_files)
rws=[r for r in rows(prot_r/'report_index.csv') if r.get('status')=='completed']; assert rws,'no completed confirmatory rows'
r0=rws[0]; rdir=Path(r0['report_dir']) if r0.get('report_dir') else (PROT_REAL/Path(r0['run_id']))
need(rdir,['config.json','metrics.json','fold_metrics.csv','fold_splits.csv','predictions.csv','spatial_compatibility_report.json','interpretability_summary.json'])
cfg=j(rdir/'config.json'); met=j(rdir/'metrics.json')
assert cfg.get('framework_mode')=='confirmatory' and met.get('framework_mode')=='confirmatory'
assert cfg.get('canonical_run') is True and met.get('canonical_run') is True
assert cfg.get('protocol_id') and cfg.get('protocol_version') and cfg.get('suite_id')
assert cfg.get('comparison_id') in (None,'')
MODE['confirmatory']={'mode':'confirmatory','command':' '.join(shlex.quote(x) for x in real),'rc':rr['rc'],'output_root':str(PROT_REAL),'report_dir':str(rdir),'framework_mode':cfg.get('framework_mode'),'canonical_run':cfg.get('canonical_run'),'protocol_meta':bool(cfg.get('protocol_id')),'comparison_meta':bool(cfg.get('comparison_id')),'manifests':', '.join([f for f in prot_files if (prot_r/f).exists()])}
CHECK['confirmatory']=True
tree(prot_r,3)

## F) Cross-Mode Summary, Guardrails, and Final Verdict
Compare mode metadata side-by-side, run negative checks, and print pass/fail summary.


In [ ]:
# Cross-mode summary, default-root policy check, guardrails, final verdict
sys.path.insert(0,str(ROOT/'src'))
from Thesis_ML.config.paths import DEFAULT_EXPLORATORY_REPORTS_ROOT, DEFAULT_COMPARISON_REPORTS_ROOT, DEFAULT_CONFIRMATORY_REPORTS_ROOT, DEFAULT_EXPERIMENT_REPORTS_ROOT, DEFAULT_PROTOCOL_REPORTS_ROOT, OUTPUTS_ROOT
assert DEFAULT_EXPERIMENT_REPORTS_ROOT==DEFAULT_EXPLORATORY_REPORTS_ROOT
assert DEFAULT_PROTOCOL_REPORTS_ROOT==DEFAULT_CONFIRMATORY_REPORTS_ROOT
assert DEFAULT_EXPLORATORY_REPORTS_ROOT==OUTPUTS_ROOT/'reports'/'exploratory'
assert DEFAULT_COMPARISON_REPORTS_ROOT==OUTPUTS_ROOT/'reports'/'comparisons'
assert DEFAULT_CONFIRMATORY_REPORTS_ROOT==OUTPUTS_ROOT/'reports'/'confirmatory'
CHECK['default_roots']=True

assert MODE['exploratory']['framework_mode']=='exploratory'
assert MODE['locked_comparison']['framework_mode']=='locked_comparison'
assert MODE['confirmatory']['framework_mode']=='confirmatory'
assert MODE['exploratory']['canonical_run'] is False
assert MODE['locked_comparison']['canonical_run'] is False
assert MODE['confirmatory']['canonical_run'] is True
assert MODE['exploratory']['protocol_meta'] is False and MODE['exploratory']['comparison_meta'] is False
assert MODE['locked_comparison']['comparison_meta'] is True and MODE['locked_comparison']['protocol_meta'] is False
assert MODE['confirmatory']['protocol_meta'] is True and MODE['confirmatory']['comparison_meta'] is False
CHECK['separation']=True

bad_cmp=run([*CMP_CMD,'--comparison',str(COMPARISON),'--all-variants','--reports-root',str(CMP_DRY),'--dry-run','--target','coarse_affect'])
assert bad_cmp['rc']!=0 and 'unrecognized arguments' in bad_cmp['err'].lower()
bad_prot=run([*PROT_CMD,'--protocol',str(PROTOCOL),'--all-suites','--reports-root',str(PROT_DRY),'--dry-run','--model','ridge'])
assert bad_prot['rc']!=0 and 'unrecognized arguments' in bad_prot['err'].lower()
CHECK['guardrails']=True

summary=[MODE['exploratory'],MODE['locked_comparison'],MODE['confirmatory']]
if pd is not None:
    display(pd.DataFrame(summary)[['mode','rc','output_root','framework_mode','canonical_run','protocol_meta','comparison_meta','manifests','report_dir']])
else:
    print(json.dumps(summary,indent=2))

verdict={
    'exploratory mode validated':CHECK.get('exploratory',False),
    'comparison mode validated':CHECK.get('comparison',False),
    'confirmatory mode validated':CHECK.get('confirmatory',False),
    'framework mode metadata separation validated':CHECK.get('separation',False),
    'artifact separation validated':CHECK.get('separation',False),
    'output-root separation validated':CHECK.get('default_roots',False),
    'guardrail rejection validated':CHECK.get('guardrails',False),
}
for k,v in verdict.items(): print(f"[{'PASS' if v else 'FAIL'}] {k}")
fail=[k for k,v in verdict.items() if not v]
if fail: raise AssertionError('Failed checks: '+', '.join(fail))
print('
All Phase 2 framework-mode checks passed.')